In [ ]:
"""
Figure 6B-D: Synapse-to-tracheole proximity by brain region and neurotransmitter.

Inputs: Princeton FAFB v783 presynaptic contacts, FlyWire neuron annotations,
and CATMAID nodes for 117 tracheoles. Skeleton edges are sampled every 100 nm.
Distances, contact-weighted plotting samples, and Kruskal-Wallis tests use the
same settings as the analysis notebook. Laminae are excluded; NT score >= 0.5.
"""

from pathlib import Path
from collections import Counter
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import kruskal


# Inputs and parameters
folder = Path.home() / "enter your path"
syn_csv = folder / "fafb_v783_princeton_synapse_table.csv"
nt_csv = folder / "neurons.csv"
trach_csv = folder / "catmaid_nodes_all_117_tracheoles.csv.gz"
out_dir = folder / "fig6"

nt_order = ["ACH", "GABA", "GLUT", "DA", "SER", "OCT"]
nt_colors = {"ACH": "#2020df", "DA": "#963939", "GABA": "#df2020", "GLUT": "#df9c20", "OCT": "#107010", "SER": "#701070"}
exclude = {"LA_L", "LA_R"}
nt_min_score = 0.5
join_chunk, dist_chunk = 500_000, 250_000
workers, trach_step_nm, min_trach_nodes = -1, 100.0, 101
coord_mode = "auto"
sample_region_n, sample_region_nt_n, sample_overshoot = 5_000, 2_000, 1.25
sample_super_n, sample_super_nt_n = 10_000, 5_000
min_sample_n, plot_qmax, seed = 20, 0.995, 20260806

superregions = {
    "OL": ["ME_L", "ME_R", "AME_L", "AME_R", "LO_L", "LO_R", "LOP_L", "LOP_R"],
    "MB": ["MB_CA_R", "MB_CA_L", "MB_PED_L", "MB_PED_R", "MB_VL_R", "MB_VL_L", "MB_ML_R", "MB_ML_L"],
    "CX": ["FB", "EB", "PB", "NO"],
    "LX": ["BU_L", "BU_R", "LAL_R", "LAL_L", "GA_R", "GA_L"],
    "VLNP": ["AOTU_R", "AOTU_L", "PVLP_L", "AVLP_L", "PVLP_R", "AVLP_R", "PLP_L", "PLP_R", "WED_L", "WED_R"],
    "LH": ["LH_L", "LH_R"], "SNP": ["SLP_L", "SLP_R", "SIP_L", "SIP_R", "SMP_R", "SMP_L"],
    "INP": ["CRE_L", "CRE_R", "ICL_L", "SCL_L", "ICL_R", "SCL_R", "IB_L", "IB_R", "ATL_R", "ATL_L"],
    "AL": ["AL_L", "AL_R"],
    "VMNP": ["VES_L", "VES_R", "EPA_R", "EPA_L", "GOR_L", "GOR_R", "SPS_L", "SPS_R", "IPS_R", "IPS_L"],
    "PENP": ["SAD", "AMMC_R", "AMMC_L", "FLA_R", "FLA_L", "CAN_R", "CAN_L", "PRW"],
    "GNG": ["GNG"], "OCG": ["OCG"],
}
region_to_super = {region: name for name, members in superregions.items() for region in members}


def column(columns, candidates):
    names = {re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_"): c for c in columns}
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]+", "_", candidate.lower()).strip("_")
        if key in names:
            return names[key]
    raise KeyError(f"Missing column; expected one of {candidates}")


def roots(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).replace(
        {"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})


def labels(series):
    return series.astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})


def load_nt():
    cols = pd.read_csv(nt_csv, nrows=0).columns
    rc = column(cols, ["root_id", "root", "flywire_root_id", "segment_id"])
    tc = column(cols, ["nt_type", "transmitter", "neurotransmitter"])
    sc = column(cols, ["nt_type_score", "transmitter_prob", "transmitter_probability", "nt_score"])
    table = pd.read_csv(nt_csv, usecols=[rc, tc, sc], dtype={rc: "string"}, low_memory=False)
    table = table.rename(columns={rc: "root_id", tc: "nt_type", sc: "nt_type_score"})
    table["root_id"] = roots(table["root_id"])
    table["nt_type"] = labels(table["nt_type"])
    table["nt_type_score"] = pd.to_numeric(table["nt_type_score"], errors="coerce")
    table = table.dropna(subset=["root_id"])
    if table.duplicated("root_id").any():
        table = table.sort_values(["root_id", "nt_type_score"], ascending=[True, False], na_position="last")
        table = table.drop_duplicates("root_id", keep="first")
    return table.reset_index(drop=True)


def sampling_counts(nt):
    # Raw root IDs here follow the original NT-counting pass.
    cols = pd.read_csv(syn_csv, nrows=0).columns
    rc = column(cols, ["pre_root_id_720575940", "pre_root_id", "presynaptic_root_id", "pre_pt_root_id"])
    nc = column(cols, ["neuropil", "brain_region", "region", "roi"])
    counts, nt_counts = Counter(), Counter()
    for part in pd.read_csv(syn_csv, usecols=[rc, nc], dtype={rc: "string", nc: "string"}, chunksize=join_chunk):
        part = part.rename(columns={rc: "pre_root_id", nc: "neuropil"})
        part["pre_root_id"] = roots(part["pre_root_id"])
        part["neuropil"] = labels(part["neuropil"]).fillna("<MISSING>")
        merged = part.merge(nt, how="left", left_on="pre_root_id", right_on="root_id", validate="many_to_one")
        counts.update(merged["neuropil"].value_counts(dropna=False).to_dict())
        good = merged["nt_type"].notna() & merged["nt_type_score"].ge(nt_min_score)
        nt_counts.update({tuple(map(str, key)): int(n) for key, n in
                          merged.loc[good, ["neuropil", "nt_type"]].value_counts().items()})
    return counts, nt_counts


def tracheole_points():
    raw = pd.read_csv(trach_csv, low_memory=False)
    cols = raw.columns
    fields = {
        "skeleton_id": ["skeleton_id", "new_skeleton_id", "skid", "neuron"],
        "node_id": ["node_id", "treenode_id", "id"],
        "parent_id": ["parent_id", "parent_node_id", "parent"],
        "x_nm": ["x_nm", "x"], "y_nm": ["y_nm", "y"], "z_nm": ["z_nm", "z"],
    }
    nodes = pd.DataFrame({name: pd.to_numeric(raw[column(cols, candidates)], errors="coerce")
                          for name, candidates in fields.items()})
    nodes = nodes.dropna(subset=["skeleton_id", "node_id", "x_nm", "y_nm", "z_nm"])
    nodes[["skeleton_id", "node_id"]] = nodes[["skeleton_id", "node_id"]].astype(np.int64)
    nodes["parent_id"] = nodes["parent_id"].astype("Int64")
    nodes = nodes.drop_duplicates(["skeleton_id", "node_id"]).reset_index(drop=True)
    keep = nodes.groupby("skeleton_id").size().loc[lambda s: s >= min_trach_nodes].index
    nodes = nodes.loc[nodes["skeleton_id"].isin(keep)]
    if nodes.empty:
        raise ValueError("No tracheoles meet the node-count threshold")

    parts = []
    for _, group in nodes.groupby("skeleton_id", sort=False):
        xyz = group[["x_nm", "y_nm", "z_nm"]].to_numpy(np.float64)
        lookup = dict(zip(group["node_id"].to_numpy(np.int64), xyz))
        root_nodes = group.loc[group["parent_id"].isna(), ["x_nm", "y_nm", "z_nm"]]
        if len(root_nodes):
            parts.append(root_nodes.to_numpy(np.float64))
        for row in group.itertuples(index=False):
            if pd.isna(row.parent_id):
                continue
            parent = lookup.get(int(row.parent_id))
            if parent is None:
                continue
            child = np.array([row.x_nm, row.y_nm, row.z_nm], dtype=np.float64)
            vector = child - parent
            length = float(np.linalg.norm(vector))
            if not np.isfinite(length):
                continue
            if length == 0:
                parts.append(child.reshape(1, 3))
                continue
            intervals = max(int(math.ceil(length / trach_step_nm)), 1)
            fractions = np.arange(1, intervals + 1, dtype=np.float64) / intervals
            parts.append(parent[None, :] + fractions[:, None] * vector[None, :])
    return np.vstack(parts).astype(np.float32, copy=False)


def root_encoding(root_col, nt):
    probe = pd.read_csv(syn_csv, usecols=[root_col], dtype={root_col: "string"}, nrows=20_000)[root_col]
    match = re.search(r"(720575940)$", str(root_col)) or re.search(r"root_id_(\d{9})", str(root_col))
    prefix = match.group(1) if match else None
    candidates = [("raw", None)] + ([("header_prefix", prefix)] if prefix else [])
    known = set(nt["root_id"].astype(str))
    scores = []
    for mode, head in candidates:
        values = transform_roots(probe, mode, head)
        unique = set(values.dropna().astype(str))
        scores.append(len(unique & known) / len(unique) if unique else 0.0)
    best = int(np.argmax(scores))
    return candidates[best]


def transform_roots(series, mode, prefix):
    result = roots(series)
    if mode == "header_prefix":
        digits = 18 - len(prefix)
        short = result.notna() & result.str.fullmatch(rf"\d{{1,{digits}}}")
        result.loc[short] = prefix + result.loc[short].str.zfill(digits)
    return result


def coordinate_scale(coords, points, tree):
    probe = pd.read_csv(syn_csv, usecols=coords, nrows=20_000, low_memory=False)
    xyz = probe[coords].apply(pd.to_numeric, errors="coerce").dropna().to_numpy(np.float64)
    modes = [coord_mode] if coord_mode != "auto" else ["nm", "voxel_4_4_40"]
    bounds = (points.astype(np.float64).min(axis=0), points.astype(np.float64).max(axis=0))
    candidates = []
    for mode in modes:
        converted = xyz if mode == "nm" else xyz * np.array([4.0, 4.0, 40.0])[None, :]
        inside = np.all(converted >= bounds[0] - 50_000, axis=1) & np.all(converted <= bounds[1] + 50_000, axis=1)
        distance_nm = tree.query(converted.astype(np.float32), k=1, workers=-1)[0]
        candidates.append((float(inside.mean()), float(np.median(distance_nm) / 1000.0), mode))
    return min(candidates, key=lambda row: (-row[0], row[1]))[2]


def sample_prob(count, target):
    return min(1.0, target * sample_overshoot / count) if count is not None and count > 0 else 0.0


def cap_samples(frame, groups, target, random_seed):
    if frame.empty:
        return frame
    rng = np.random.default_rng(random_seed)
    shuffled = frame.iloc[rng.permutation(len(frame))]
    return shuffled.groupby(groups, dropna=False, sort=False, group_keys=False).head(target).reset_index(drop=True)


def measure(nt, counts, nt_counts):
    points = tracheole_points()
    tree = cKDTree(points)
    cols = pd.read_csv(syn_csv, nrows=0).columns
    rc = column(cols, ["pre_root_id_720575940", "pre_root_id", "presynaptic_root_id", "pre_pt_root_id"])
    nc = column(cols, ["neuropil", "brain_region", "region", "roi"])
    coords = [column(cols, options) for options in (
        ["pre_x", "presynaptic_x", "pre_pt_position_x"],
        ["pre_y", "presynaptic_y", "pre_pt_position_y"],
        ["pre_z", "presynaptic_z", "pre_pt_position_z"])]
    id_mode, id_prefix = root_encoding(rc, nt)
    unit = coordinate_scale(coords, points, tree)
    rng = np.random.default_rng(seed)
    all_parts, nt_parts = [], []
    exact, nt_exact = Counter(), Counter()
    totals = Counter()

    reader = pd.read_csv(syn_csv, usecols=[rc, nc, *coords], dtype={rc: "string", nc: "string"},
                         chunksize=dist_chunk, low_memory=False)
    for chunk in reader:
        part = chunk.rename(columns={rc: "pre_root_id_raw", nc: "neuropil",
                                     coords[0]: "pre_x_raw", coords[1]: "pre_y_raw", coords[2]: "pre_z_raw"})
        part["neuropil"] = labels(part["neuropil"]).fillna("<MISSING>")
        part["pre_root_id"] = transform_roots(part["pre_root_id_raw"], id_mode, id_prefix)
        xyz = part[["pre_x_raw", "pre_y_raw", "pre_z_raw"]].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
        valid = np.all(np.isfinite(xyz), axis=1)
        d = np.full(len(part), np.nan, dtype=np.float64)
        if valid.any():
            converted = xyz[valid] if unit == "nm" else xyz[valid] * np.array([4.0, 4.0, 40.0])[None, :]
            d[valid] = tree.query(converted.astype(np.float32), k=1, workers=workers)[0] / 1000.0
        part["distance_to_nearest_trachea_um"] = d
        merged = part.merge(nt, how="left", left_on="pre_root_id", right_on="root_id", validate="many_to_one")
        usable = merged.loc[merged["distance_to_nearest_trachea_um"].notna(),
                            ["neuropil", "pre_root_id", "pre_x_raw", "pre_y_raw", "pre_z_raw",
                             "distance_to_nearest_trachea_um"]].copy()
        good = merged["nt_type"].notna() & merged["nt_type_score"].ge(nt_min_score)
        high_nt = merged.loc[good & merged["distance_to_nearest_trachea_um"].notna(),
                            ["neuropil", "pre_root_id", "pre_x_raw", "pre_y_raw", "pre_z_raw",
                             "nt_type", "nt_type_score", "distance_to_nearest_trachea_um"]].copy()

        exact.update(usable["neuropil"].value_counts(dropna=False).to_dict())
        nt_exact.update({tuple(map(str, k)): int(n) for k, n in
                         high_nt[["neuropil", "nt_type"]].value_counts().items()})
        included = usable["neuropil"].isin(region_to_super)
        distances = usable.loc[included, "distance_to_nearest_trachea_um"]
        totals["contacts"] += len(distances)
        totals["within_20_um"] += int((distances <= 20).sum())

        if counts and len(usable):
            expected = usable["neuropil"].map(counts)
            prob = np.array([sample_prob(int(x) if pd.notna(x) else None, sample_region_n) for x in expected])
            chosen = rng.random(len(usable)) < prob
            if chosen.any():
                all_parts.append(usable.loc[chosen].copy())
        if nt_counts and len(high_nt):
            keys = zip(high_nt["neuropil"].astype(str), high_nt["nt_type"].astype(str))
            prob = np.array([sample_prob(nt_counts.get(key), sample_region_nt_n) for key in keys])
            chosen = rng.random(len(high_nt)) < prob
            if chosen.any():
                nt_parts.append(high_nt.loc[chosen].copy())

    all_sample = cap_samples(pd.concat(all_parts, ignore_index=True), ["neuropil"], sample_region_n, seed + 1)
    nt_sample = cap_samples(pd.concat(nt_parts, ignore_index=True), ["neuropil", "nt_type"],
                            sample_region_nt_n, seed + 2)
    all_exact = pd.DataFrame([{"neuropil": k, "n": v} for k, v in exact.items()])
    nt_exact = pd.DataFrame([{"neuropil": k[0], "nt_type": k[1], "n": v} for k, v in nt_exact.items()])
    return all_sample, nt_sample, all_exact, nt_exact, totals


def prepare(all_sample, nt_sample, all_exact, nt_exact):
    frames = [all_sample, nt_sample, all_exact, nt_exact]
    for i, frame in enumerate(frames):
        frame["neuropil"] = labels(frame["neuropil"]).fillna("<MISSING>").astype(str)
        frame = frame.loc[(frame["neuropil"] != "<MISSING>") & ~frame["neuropil"].str.upper().isin(exclude)].copy()
        if i in (1, 3):
            frame["nt_type"] = labels(frame["nt_type"]).fillna("<MISSING>").astype(str).str.upper()
        frames[i] = frame
    all_sample, nt_sample, all_exact, nt_exact = frames
    observed = set(all_sample["neuropil"]) | set(all_exact["neuropil"])
    unknown = observed - set(region_to_super)
    if unknown:
        raise ValueError(f"Unmapped neuropils: {sorted(unknown)}")
    for frame in frames:
        frame["super_region"] = frame["neuropil"].map(region_to_super)
    return frames


def weighted_sample(sample, exact, groups, target, random_seed):
    sizes = sample.groupby(groups, as_index=False).size().rename(columns={"size": "available_sample_n"})
    counts = exact[groups + ["n"]].rename(columns={"n": "exact_n"})
    weights = counts.merge(sizes, on=groups, how="inner", validate="one_to_one")
    weights["row_weight"] = weights["exact_n"] / weights["available_sample_n"]
    rows = sample.merge(weights[groups + ["row_weight"]], on=groups, how="inner", validate="many_to_one")
    rng, parts = np.random.default_rng(random_seed), []
    for _, part in rows.groupby(["super_region"] + (["nt_type"] if "nt_type" in rows else []),
                                dropna=False, sort=True):
        weights = pd.to_numeric(part["row_weight"], errors="coerce").to_numpy(np.float64)
        good = np.isfinite(weights) & (weights > 0)
        part, weights = part.loc[good], weights[good]
        if len(part):
            idx = rng.choice(len(part), size=target, replace=True, p=weights / weights.sum())
            parts.append(part.iloc[idx].copy())
    return pd.concat(parts, ignore_index=True)


def kruskal_summary(sample, nt_type=None):
    arrays = [group["distance_to_nearest_trachea_um"].dropna().to_numpy(np.float64)
              for _, group in sample.groupby("super_region", sort=True)]
    arrays = [values for values in arrays if len(values) >= min_sample_n]
    result = kruskal(*arrays, nan_policy="omit")
    n, k = sum(map(len, arrays)), len(arrays)
    return {"panel": "6B" if nt_type is None else "6C", "nt_type": "all" if nt_type is None else nt_type,
            "groups": k, "sample_n": n, "H": float(result.statistic), "p": float(result.pvalue),
            "epsilon_squared": (float(result.statistic) - k + 1) / (n - k)}


# Figures
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 13, "axes.titlesize": 14, "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 13,
    "axes.linewidth": 0.65, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 3, "ytick.major.size": 3, "svg.fonttype": "none",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
})


def style_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(0.65)
    ax.tick_params(direction="out", width=0.6, length=3, pad=2)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color="#D9D9D9", linewidth=0.45, alpha=0.7)


def style_violin(violin, color):
    for body in violin.get("bodies", []):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.58)
        body.set_linewidth(0.55)
    for key in ("cmedians", "cbars", "cmins", "cmaxes"):
        if key in violin:
            violin[key].set_color("#202020")
            violin[key].set_linewidth(0.8)


def xmax(arrays):
    return float(np.quantile(np.concatenate([a[np.isfinite(a)] for a in arrays if len(a)]), plot_qmax))


def fig6b(sample):
    groups = sorted(sample["super_region"].dropna().unique())
    arrays = [sample.loc[sample["super_region"] == g, "distance_to_nearest_trachea_um"].dropna().to_numpy(float)
              for g in groups]
    groups, arrays = zip(*[(g, a) for g, a in zip(groups, arrays) if len(a)])
    fig, ax = plt.subplots(figsize=(7.0, max(3.2, 0.34 * len(arrays) + 1.15)))
    pos = np.arange(1, len(arrays) + 1)
    violins = ax.violinplot(arrays, positions=pos, vert=False, widths=0.78, showmeans=False,
                            showmedians=False, showextrema=False, points=100, bw_method="scott")
    style_violin(violins, "#3F6F8F")
    ax.scatter([np.median(a) for a in arrays], pos, s=32, facecolor="#202020", edgecolor="white",
               linewidth=0.35, zorder=4, label="Median")
    ax.set_yticks(pos, groups)
    ax.invert_yaxis()
    ax.set_xlim(0, xmax(arrays))
    ax.set_xlabel("Distance to nearest tracheole point (µm)")
    ax.set_title("All synaptic contacts across super-regions", loc="left", fontweight="semibold", pad=7)
    style_axis(ax)
    ax.legend(frameon=False, loc="lower right", fontsize=15, markerscale=1.65, handletextpad=0.7,
              borderaxespad=0.65, labelspacing=0.55)
    fig.tight_layout(pad=0.6)
    return fig


def fig6c(sample):
    present = [nt for nt in nt_order if nt in set(sample["nt_type"].astype(str))]
    fig, axes = plt.subplots(3, 2, figsize=(10.2, 11.2))
    for ax, nt in zip(axes.ravel(), present):
        subset = sample.loc[sample["nt_type"] == nt]
        groups = sorted(subset["super_region"].dropna().astype(str).unique())
        arrays = [subset.loc[subset["super_region"].astype(str) == g,
                             "distance_to_nearest_trachea_um"].dropna().to_numpy(float) for g in groups]
        groups, arrays = zip(*[(g, a) for g, a in zip(groups, arrays) if len(a)])
        pos = np.arange(1, len(arrays) + 1)
        v = ax.violinplot(arrays, positions=pos, vert=False, widths=0.76, showmeans=False,
                          showmedians=True, showextrema=False, points=100, bw_method="scott")
        style_violin(v, nt_colors.get(nt, "#7A7A7A"))
        ax.set_yticks(pos, groups, fontsize=11)
        ax.invert_yaxis()
        ax.set_xlim(0, xmax(arrays))
        ax.set_title(nt, loc="left", fontweight="semibold", pad=5)
        ax.set_xlabel("Distance (µm)")
        style_axis(ax)
    for ax in axes.ravel()[len(present):]:
        ax.axis("off")
    fig.suptitle("Neurotransmitter-specific synapse-to-trachea distances across super-regions",
                 fontsize=14, fontweight="semibold", x=0.08, ha="left", y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.975), h_pad=1.0, w_pad=1.0)
    return fig


def fig6d(summary):
    pivot = summary.pivot_table(index="super_region", columns="nt_type", values="sample_median_um", aggfunc="first")
    ordered = [nt for nt in nt_order if nt in pivot.columns]
    pivot = pivot[ordered + sorted(c for c in pivot.columns if c not in ordered)]
    pivot = pivot.loc[sorted(pivot.index.astype(str))]
    fig, ax = plt.subplots(figsize=(6.7, max(4.2, 0.22 * len(pivot) + 1.45)))
    matrix = pivot.to_numpy(float)
    im = ax.imshow(matrix, aspect="auto", cmap="viridis", interpolation="nearest")
    ax.set_xticks(np.arange(len(pivot.columns)), pivot.columns)
    ax.set_yticks(np.arange(len(pivot.index)), pivot.index, fontsize=11)
    ax.set_xlabel("Predicted neurotransmitter")
    ax.set_title("Median distance by super-region and neurotransmitter", loc="left", fontweight="semibold", pad=7)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    bar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
    bar.outline.set_linewidth(0.5)
    bar.ax.tick_params(width=0.5, length=2)
    bar.set_label("Median distance to nearest tracheole (µm)")
    values = matrix[np.isfinite(matrix)]
    mid = float(np.nanmedian(values)) if len(values) else 0.0
    for row, col in np.ndindex(matrix.shape):
        val = matrix[row, col]
        if np.isfinite(val):
            ax.text(col, row, f"{val:.1f}", ha="center", va="center", fontsize=10.5,
                    color="white" if val <= mid else "black")
    fig.tight_layout(pad=0.6)
    return fig


# Run analysis and save manuscript figures
out_dir.mkdir(parents=True, exist_ok=True)
nt = load_nt()
counts, nt_counts = sampling_counts(nt)
all_sample, nt_sample, all_exact, nt_exact, totals = measure(nt, counts, nt_counts)
all_sample, nt_sample, all_exact, nt_exact = prepare(all_sample, nt_sample, all_exact, nt_exact)
all_super = weighted_sample(all_sample, all_exact, ["neuropil"], sample_super_n, seed + 1)
nt_super = weighted_sample(nt_sample, nt_exact, ["neuropil", "nt_type"], sample_super_nt_n, seed + 2)

median_nt = nt_super.groupby(["super_region", "nt_type"], as_index=False)["distance_to_nearest_trachea_um"].median()
median_nt = median_nt.rename(columns={"distance_to_nearest_trachea_um": "sample_median_um"})

for panel, plot in (("6b", fig6b(all_super)), ("6c", fig6c(nt_super)), ("6d", fig6d(median_nt))):
    plot.savefig(out_dir / f"fig{panel}.svg", format="svg", bbox_inches="tight", pad_inches=0.04)
    plt.show()

stats = [kruskal_summary(all_super)]
for nt_name in nt_order:
    subset = nt_super.loc[nt_super["nt_type"] == nt_name]
    if not subset.empty:
        stats.append(kruskal_summary(subset, nt_name))
stats = pd.DataFrame(stats)
stats.to_csv(out_dir / "fig6_stats.csv", index=False)
median_nt.to_csv(out_dir / "fig6d_medians.csv", index=False)
print(stats.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print(f"Contacts in mapped neuropils: {totals['contacts']:,}; within 20 µm: "
      f"{totals['within_20_um'] / totals['contacts']:.1%}")
